# Step 0 · Query diagnostics

This notebook documents **how the search criteria were chosen**: which CPC codes, IPC codes and keywords define the flying-drones query, and why these and not others.

It is separate from `1-DataPreparation`, which contains only the final lists and produces the dataset. The lists defined here are the **initial** ones (the query as it stood before testing) plus the alternatives that were tested, so the results below stay reproducible when the final lists change.

**Method.** Every code and keyword is a candidate and passes up to three checks:

| Check | Question | Used for |
|---|---|---|
| 1. Definition | Does the candidate cover unmanned aircraft only? | Classification codes |
| 2. Count | How many families does it add, and is it redundant? | Codes and keywords |
| 3. Sample | Is what it adds relevant? (random sample labelled against the sector definition) | Keywords, and codes whose definition leaves a doubt |

A classification code is assigned by an examiner who read the document, so checks 1 and 2 are usually enough. A keyword is only a string match: two keywords that look alike in a count can differ widely in relevance, so keywords always need check 3. Decision rules are fixed before looking at the results.

**Conventions.** Counts are DOCDB families, PATSTAT PROD, filing years 2000–2024. Relevance labels come from the LLM validator of `2-DataValidation.ipynb` (title + longest English abstract); the answers are cached in `validation/llm_cache.jsonl`, so re-running this notebook makes no new LLM calls as long as the extraction is unchanged.

**Output columns** (in Italian): `famiglie` = families retrieved · `solo_questo` / `solo_questa_keyword` = retrieved by this candidate only · `senza_classe` = without any selected class · `pertinenti` = relevant · `incerti` = undecided by the LLM · `quota_pertinenti` = share of relevant families, with its 95% Wilson interval (`IC95_min`, `IC95_max`).

## 1. Set-up and sector definition

`DRONE_DESCRIPTION` is the written definition of the sector. All criteria are derived from it, and it is the text given to the LLM that labels the samples.

In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
%pip install -q pandas numpy sqlalchemy 

import hashlib
from pathlib import Path

import numpy as np



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
ENV = "PROD"                  # all counts in this notebook refer to PROD
YEARS = (2000, 2024)          # filing years

RANDOM_STATE = 42


DRONE_DESCRIPTION = """
Flying drones: aircraft that fly without a human pilot on board, remotely
piloted or autonomous, of any size, configuration and end use.

In scope:
1. The vehicle: airframe, wings and rotors, propulsion and power supply,
   undercarriage.
2. Flight control, navigation, obstacle avoidance, command and data links,
   ground control stations, swarm coordination.
3. Dedicated infrastructure: launch, landing, recovery, charging or battery
   swap, docking, storage and transport.
4. Airspace and traffic management for unmanned aircraft.
5. Mission equipment and applications, when the invention concerns the
   aircraft itself or the way it is operated in flight (e.g. payload release
   for delivery, spraying controlled through the flight path).

Out of scope:
- Generic technologies for which a drone is only one of several possible
  carriers or uses.
- Inventions that only use data or services provided by a drone (e.g.
  analysis of images taken from a drone).
- Aircraft with a pilot on board.
- Ground-based counter-drone systems (a drone that intercepts other drones
  is in scope).
- Underwater, surface and ground unmanned vehicles.
- Guided missiles and projectiles.

Test for doubtful cases: if the drone is removed, does the invention still
make sense? If yes, it is out of scope.
""".strip()


In [4]:
patstat = PatstatClient(env=ENV) # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()  # SQLAlchemy session used by all ORM queries below

## 2. Initial search criteria

The query as it stood before testing: 22 keywords, one IPC prefix and three CPC prefixes, combined with OR. Codes are matched by prefix (`B64U` selects the whole subclass); keywords are matched as substrings of titles and abstracts.

Link to the sector definition: `B64U` covers the vehicle, its controls, its infrastructure and its uses; `B64C39/024` is where drones were classified before `B64U` existed (2023); `G08G5/57` covers airspace and traffic management for unmanned aircraft.

In [5]:
DRONE_KEYWORDS = (
    # Core UAV terminology
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "unmanned aircraft system",
    "unmanned air vehicle",
    "uncrewed aircraft",
    "remotely piloted aircraft",

    # UAV configurations
    "quadcopter",
    "quadrotor",
    "quadrocopter",
    "multicopter",
    "hexacopter",
    "octocopter",
    "micro air vehicle",
    "fixed-wing uav",
    "vtol uav",

    # UAV-specific technologies
    "uav navigation",
    "uav flight control",
    "uav communication",
    "uav swarm",
    "uav propulsion",

    # Broad term
    "drone",
)

DRONE_IPC_CODES = (
    "B64U",
)
DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
    "G08G5/57",
)

## 3. Initial extraction

Applications retrieved by at least one criterion (keywords OR IPC OR CPC), with their English title and abstract. `df_drone` is the table on which the keyword checks below are computed.

In [6]:
def keyword_condition(terms, patterns=()):
    """OR-condition: a term (substring) or a pattern (regular expression)
    appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
        *(title.regexp_match(p) for p in patterns),
        *(abstract.regexp_match(p) for p in patterns),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Application-level fields retrieved by the search queries.
# These fields describe each application; they do not change the search criteria.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,              # Unique application identifier
    TLS201_APPLN.docdb_family_id,       # Links applications belonging to the same DOCDB family
    TLS201_APPLN.appln_auth,            # Filing authority or patent office
    TLS201_APPLN.appln_nr,              # Application number
    TLS201_APPLN.appln_kind,            # Application kind code
    TLS201_APPLN.appln_filing_date,     # Filing date of this application
    TLS201_APPLN.earliest_filing_year,  # Earliest filing year associated with this application
    TLS201_APPLN.granted,               # Grant indicator: "Y" means granted
)

#3 TYPES OF QUERIES

#PROF BASELINE, MODIFIES:
#in each -> .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))

def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

# COMBINE THE SEARCH RESULTS

# Select application IDs found by at least one search strategy:
# keywords OR IPC codes OR CPC codes.
# UNION removes duplicate IDs across the three searches.

appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(
        query_by_ipc(appln_id),
        query_by_cpc(appln_id),
    )
    .subquery()
)

#each query (keyword,CPC,IPD) retrieve only appln_id foreach application

#with this function we retrieve all the others attributes
def query_details(ids_table):
    return (
        db.query(
            *APPLN_COLUMNS,
            TLS202_APPLN_TITLE.appln_title,
            TLS203_APPLN_ABSTR.appln_abstract,
        )
        .select_from(TLS201_APPLN)
        .join(
            ids_table,
            TLS201_APPLN.appln_id == ids_table.c.appln_id,
        )
        .outerjoin(
            TLS202_APPLN_TITLE,
            and_(
                TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
                func.lower(func.trim(TLS202_APPLN_TITLE.appln_title_lg)) == "en",
            ),
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            and_(
                TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
                func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en",
            ),
        )
        .distinct()
    )

# Prepare the DataFrame with one row per patent application (appln_id).
# Replace missing titles and abstracts with empty strings.
# If multiple rows exist for the same application, keep the longest abstract.
def one_application(df):
    columns = [c.key for c in APPLN_COLUMNS]
    columns += ["appln_title", "appln_abstract"]
    df = df.reindex(columns=columns).copy()

    for column in ["appln_title", "appln_abstract"]:
        df[column] = df[column].fillna("").astype(str).str.strip()

    df["_abstract_length"] = df.appln_abstract.str.len()

    return (
        df.sort_values(
            ["appln_id", "_abstract_length", "appln_abstract", "appln_title"],
            ascending=[True, False, True, True],
        )
        .drop_duplicates("appln_id")
        .drop(columns="_abstract_length")
        .reset_index(drop=True)
    )

q_drone = query_details(drone_ids)
df_drone = one_application(patstat.df(q_drone))

df_drone.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,2442,38192468,EP,06838485,A,2006-11-29,2005,Y,ELECTRIC MOTOR ASSISTED TAKEOFF DEVICE FOR AN ...,"A takeoff assist device for an air vehicle, su..."
1,4232,39680951,EP,08290407,A,2008-04-29,2008,N,Method for managing the risks associated with ...,A method of managing the risk of an increased ...
2,9911,39386360,WO,2007025984,W,2007-12-20,2006,N,SOLVENT ADDITION AND REMOVAL IN THE HYDROGENAT...,Disclosed are methods for treating catmint oil...
3,9913,39273109,WO,2007025987,W,2007-12-20,2006,N,PRODUCTION OF DIHYDRONEPETALACTONE BY HYDROGEN...,This invention provides a process for producin...
4,9915,39386413,WO,2007025988,W,2007-12-20,2006,N,HYDROGENATION OF CATMINT OIL,Disclosed are methods for treating catmint oil...
5,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
6,28800,39609207,WO,2007069569,W,2007-05-23,2006,N,DUAL-USE MODULAR PROPULSION SURVEILLANCE VEHIC...,The present invention provides a system for re...
7,28822,39609212,WO,2007073846,W,2007-07-19,2006,N,AERODYNAMIC INTEGRATION OF A PAYLOAD CONTAINER...,A vertical takeoff and landing (VTOL) rotary-w...
8,36005,37810003,WO,2008000133,W,2008-01-16,2007,N,APPARATUS FOR A HIVE,A trap(1) for attachment to a hive entrance co...
9,39269,39636744,WO,2008051476,W,2008-01-18,2007,N,ROTARYCRAFT POWER AND PROPULSION SYSTEM,A rotary wing vehicle includes a body structur...


## 4. CPC codes

**Candidates.** The three prefixes of the initial query, plus three alternatives: `B64C39/02` (parent group of `B64C39/024`), `G05D2109/20` (control of aircraft) and `G08G5/0069` (the code that `G08G5/57` replaced in 2025).

**Rule.** A code is used only if its official definition covers unmanned aircraft exclusively. The count shows what each code contributes: `solo_questo` is the number of families that carry this code and none of the other candidates, i.e. what is lost if the code is removed.

**Decision.**

| Code | Decision | Evidence |
|---|---|---|
| `B64U` | Kept in full | Dedicated subclass: 32,306 families with a main group (10–80), 28,401 with an additional code (2101 uses, 2201 flight controls) |
| `B64C39/024` | Kept | 7,364 families; 992 carry no other candidate code, so they would be lost without it |
| `G08G5/57` | Kept | 3,507 families, 1,446 with no other candidate code |
| `G08G5/0069` | Not needed | No rows: fully reclassified into `G08G5/57` in this PATSTAT edition |
| `B64C39/02` (other than /024) | Excluded | Aircraft "characterised by special use", not limited to unmanned aircraft; would add 3,175 families |
| `G05D2109/20` | Excluded | Control of aircraft in general, including piloted ones; would add 1,197 families |

The whole classes around these groups (`B64`, `G05D1/`, `G08G5/`) were tested earlier as an extension of the query: only 17% of the families they added were relevant (`validation/enrichment_decision.csv`, TEST environment).

**Limits.** The two exclusions rest on the definitions: the relevance of the families they would add was not measured. The 4,044 families that carry only an additional `B64U` code have their main classification elsewhere and may include inventions with an incidental drone use.

In [7]:
#CONTROLLI CPC
CANDIDATES = ("B64U", "B64C39/02", "G08G5/57", "G08G5/0069", "G05D2109/20")

sym = func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "")
q = (db.query(TLS201_APPLN.docdb_family_id, sym.label("cpc"))
     .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
     .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
     .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, CANDIDATES))
     .distinct())
hits = patstat.df(q)

def group(s):
    if s.startswith("B64U"):
        return "B64U 2101/2201 (aggiuntivi)" if int(s[4:].split("/")[0]) >= 2000 else "B64U 10-80"
    if s.startswith("B64C39/024"):
        return "B64C39/024"
    for c in ("B64C39/02", "G08G5/57", "G08G5/0069", "G05D2109/20"):
        if s.startswith(c):
            return c + (" (escluso /024)" if c == "B64C39/02" else "")

hits["group"] = hits["cpc"].map(group)
fam = hits.groupby("docdb_family_id")["group"].agg(set)
pd.DataFrame([
    {"codice": g,
     "famiglie": int(fam.apply(lambda x: g in x).sum()),
     "solo_questo": int(fam.apply(lambda x: x == {g}).sum())}
    for g in sorted(hits["group"].unique())
])



,codice,famiglie,solo_questo
0,B64C39/02 (escluso /024),6504,3175
1,B64C39/024,7364,992
2,B64U 10-80,32306,10745
3,B64U 2101/2201 (aggiuntivi),28401,4044
4,G05D2109/20,2004,1197
5,G08G5/57,3507,1446


## 5. IPC codes

**Role of IPC.** To reach documents that carry no CPC classification. The count therefore compares each IPC candidate with the CPC selection: `non_trovate_da_CPC_scelti` = families not found by the selected CPC codes; `senza_alcun_CPC` = families with no CPC code at all.

**Candidates.** `B64U` (initial query), `B64C39/02` (the IPC place for drones before `B64U`; IPC has no `/024` subgroup) and `G08G5/57` (the group selected in CPC).

**Rule.** Same as CPC. In addition: a candidate that adds almost nothing to the CPC selection is redundant; one that adds many families from a non-specific class needs a further check.

**Decision.**

| Code | Decision | Evidence |
|---|---|---|
| `B64U` | Kept in full | 37,896 families with a main group, 16,668 of them not found by the selected CPC codes; about three quarters of these have no CPC code at all, so IPC is not a duplicate of CPC |
| `G08G5/57` | Added | Only 148 new families (the group exists in IPC from 2025), added for consistency with CPC |
| `B64C39/02` | See next step | 32,192 families, 18,353 not found by the selected CPC codes: too many to decide on the definition alone |

In [8]:
#CONTROLLI IPC
IPC_CANDIDATES = ("B64U", "B64C39/02", "G08G5/57")

ipc_sym = func.replace(TLS209_APPLN_IPC.ipc_class_symbol, " ", "")
ipc_base = (db.query(TLS201_APPLN.docdb_family_id.label("fam"), ipc_sym.label("ipc"))
            .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
            .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
            .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, IPC_CANDIDATES))
            .distinct())
ipc_hits = patstat.df(ipc_base)
ipc_hits.columns = ["fam", "ipc"]

# Famiglie già trovate dai codici CPC scelti
cpc_sel = set(patstat.df(
    db.query(TLS201_APPLN.docdb_family_id)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
    .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    .distinct()).iloc[:, 0])

# Famiglie (tra quelle trovate dall'IPC) che hanno almeno un codice CPC qualsiasi
ipc_fams = ipc_base.with_entities(TLS201_APPLN.docdb_family_id.label("fam")).distinct().subquery()
any_cpc = set(patstat.df(
    db.query(TLS201_APPLN.docdb_family_id)
    .join(ipc_fams, TLS201_APPLN.docdb_family_id == ipc_fams.c.fam)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .distinct()).iloc[:, 0])

def ipc_group(s):
    if s.startswith("B64U"):
        num = s[4:].split("/")[0]
        if not num.isdigit():
            return "B64U (solo sottoclasse)"
        return "B64U indicizzazione (>=100)" if int(num) >= 100 else "B64U 10-80"
    return "B64C39/02" if s.startswith("B64C39/02") else "G08G5/57"

ipc_hits["group"] = ipc_hits["ipc"].map(ipc_group)
pd.DataFrame([
    {"codice": g,
     "famiglie": f.nunique(),
     "non_trovate_da_CPC_scelti": (~f.drop_duplicates().isin(cpc_sel)).sum(),
     "senza_alcun_CPC": (~f.drop_duplicates().isin(any_cpc)).sum()}
    for g, f in ipc_hits.groupby("group")["fam"]
])

,codice,famiglie,non_trovate_da_CPC_scelti,senza_alcun_CPC
0,B64C39/02,32192,18353,13073
1,B64U (solo sottoclasse),10,6,2
2,B64U 10-80,37896,16668,12182
3,B64U indicizzazione (>=100),16058,8045,6162
4,G08G5/57,504,148,74


### IPC `B64C39/02`: what would it really add?

The definition of `B64C39/02` is not limited to unmanned aircraft, so by the rule it is excluded. Because it is large, the cost of the exclusion is measured: how many of its families are found by no selected class, and how many of those are already retrieved by the keywords.

In [9]:
b64u_ipc = set(ipc_hits.loc[ipc_hits["group"].str.startswith("B64U"), "fam"])
c39 = set(ipc_hits.loc[ipc_hits["group"] == "B64C39/02", "fam"])

extra = c39 - cpc_sel - b64u_ipc
extra_no_cpc = extra - any_cpc
extra_with_cpc = extra & any_cpc

print("B64C39/02, famiglie aggiunte rispetto a CPC scelti + IPC B64U:", len(extra))
print("  senza alcun CPC:", len(extra_no_cpc))
print("  con CPC, ma non tra quelli scelti:", len(extra_with_cpc))

B64C39/02, famiglie aggiunte rispetto a CPC scelti + IPC B64U: 17037
  senza alcun CPC: 12501
  con CPC, ma non tra quelli scelti: 4536


In [10]:



found = extra & set(df_drone["docdb_family_id"])
print("Famiglie B64C39/02 aggiuntive:", len(extra))
print("  già nel dataset grazie alle keyword:", len(found))
print("  non trovate da nessun criterio:", len(extra - found))

Famiglie B64C39/02 aggiuntive: 17037
  già nel dataset grazie alle keyword: 12209
  non trovate da nessun criterio: 4828


**Decision: excluded.** Of the 17,037 families that `B64C39/02` would add to the selected classes, 12,209 (72%) are already in the dataset through the keywords. The exclusion leaves out at most 4,828 families, whose relevance was not measured.

## 6. Keywords: what each one contributes

**Question.** How much of the dataset depends on keywords, and on which ones?

`class_fams` = families with at least one selected class (CPC selection, IPC `B64U`, IPC `G08G5/57`). For each keyword: families retrieved, families without any selected class, and families retrieved by this keyword only (`solo_questa_keyword`). Counts use the English titles and abstracts of the extraction.

**Reading.**

- Two thirds of the dataset (125,332 of 186,320 families) enter through keywords only.
- One keyword, "unmanned aerial vehicle", brings 110,815 families that nothing else finds: 59% of the dataset.
- "drone" brings 8,675 exclusive families.
- "unmanned aircraft system" has no exclusive family (it is contained in "unmanned aircraft"): redundant.
- The seven terms that combine "uav" with a technology or configuration bring 66 exclusive families in total.

A count cannot tell whether these families are relevant: the next step samples them.

In [11]:
#CONTROLLI KEYWORDS
text = (df_drone["appln_title"] + " " + df_drone["appln_abstract"]).str.lower()
class_fams = cpc_sel | set(ipc_hits.loc[ipc_hits["group"] != "B64C39/02", "fam"])

kw_fams = {k: set(df_drone.loc[text.str.contains(k.lower(), regex=False), "docdb_family_id"])
           for k in DRONE_KEYWORDS}

all_fams = set(df_drone["docdb_family_id"])
print("Famiglie totali:", len(all_fams))
print("  con una classe scelta:", len(all_fams & class_fams))
print("  entrate solo per keyword:", len(all_fams - class_fams))

rows = []
for k, f in kw_fams.items():
    others = set().union(*(v for kk, v in kw_fams.items() if kk != k))
    rows.append({"keyword": k,
                 "famiglie": len(f),
                 "senza_classe": len(f - class_fams),
                 "solo_questa_keyword": len(f - class_fams - others)})
pd.DataFrame(rows).sort_values("famiglie", ascending=False)

Famiglie totali: 186320
  con una classe scelta: 60988
  entrate solo per keyword: 125332


,keyword,famiglie,senza_classe,solo_questa_keyword
0,unmanned aerial vehicle,148776,113567,110815
21,drone,18055,9805,8675
2,unmanned aircraft,2829,1688,1234
4,unmanned air vehicle,1606,1304,339
8,quadrotor,646,536,192
10,multicopter,514,265,207
1,unmanned aerial system,354,112,57
3,unmanned aircraft system,303,104,0
7,quadcopter,230,149,110
13,micro air vehicle,139,110,84


## 7. Relevance by origin

**Question.** Where is the noise: in the classes, in the dominant keyword, or in "drone"?

Three groups, 100 random families each, labelled by the LLM against `DRONE_DESCRIPTION`.

**Result.**

| Group | Families | Relevant |
|---|---|---|
| With a selected class | 60,988 | 0.88 (0.79–0.93), 11 undecided |
| Only "unmanned aerial vehicle" | 110,815 | 0.95 (0.89–0.98) |
| Only "drone" | 8,675 | 0.61 (0.51–0.70) |

- The largest group is also the cleanest: "unmanned aerial vehicle" is kept and needs no filter.
- The noise is concentrated in "drone".
- For the class group, most undecided cases are abstracts that describe an aircraft without saying whether it is unmanned; the examiner classified them as drones, so 0.88 is probably a lower bound. A previous draw of the same group gave 0.91: the difference is sampling variation.
- Weighted over the three groups, about 91% of the families are relevant **without any semantic filter**.

Undecided cases are excluded from the shares and still have to be labelled by hand.

In [12]:
%pip install -q langchain-core langchain-openai pydantic
%run -i ./2-DataValidation.ipynb   # uses the DRONE_DESCRIPTION defined above

def only(k):
    others = set().union(*(v for kk, v in kw_fams.items() if kk != k))
    return kw_fams[k] - class_fams - others

groups = {"Con una classe scelta": all_fams & class_fams,
          "Solo 'unmanned aerial vehicle'": only("unmanned aerial vehicle"),
          "Solo 'drone'": only("drone")}

rows, samples = [], {}
for name, fams in groups.items():
    sample = validate_sample(df_drone[df_drone["docdb_family_id"].isin(fams)], n=100)
    samples[name] = sample
    lab = sample["llm_label"]
    k, n = int((lab == "pertinente").sum()), int(lab.isin(["pertinente", "non_pertinente"]).sum())
    low, high = wilson_interval(k, n)
    rows.append({"gruppo": name, "famiglie": len(fams), "etichettate": len(sample),
                 "pertinenti": k, "incerti": int((lab == "incerto").sum()),
                 "quota_pertinenti": round(k / n, 2) if n else None,
                 "IC95_min": round(low, 2), "IC95_max": round(high, 2)})
pd.DataFrame(rows)


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Course API key:  ········


Rows without a valid family or abstract: 10344
Families labelled: 100 (new LLM calls: 100, already in cache: 0)
Rows without a valid family or abstract: 498
Families labelled: 100 (new LLM calls: 0, already in cache: 100)
Rows without a valid family or abstract: 1158
Families labelled: 100 (new LLM calls: 0, already in cache: 100)


,gruppo,famiglie,etichettate,pertinenti,incerti,quota_pertinenti,IC95_min,IC95_max
0,Con una classe scelta,60988,100,78,11,0.88,0.79,0.93
1,Solo 'unmanned aerial vehicle',110815,100,94,1,0.95,0.89,0.98
2,Solo 'drone',8675,100,59,3,0.61,0.51,0.70


## 8. The keyword "drone": substring or whole word?

The query matched "drone" as a substring, so it also retrieved words that contain it (for example the drug *dronedarone*, or *dihydronepetalactone*, visible in the first rows of the extraction above). The families retrieved only by "drone" are split into those where "drone" / "drones" is a whole word and those where it only appears inside another word; the labels of the sample above are then read separately for the two groups (no new LLM calls).

**Decision: "drone" is kept, searched as a whole word** (`\bdrones?\b`).

- Inside another word only: 1,659 families; 0 relevant out of 18 labelled. Removed.
- Whole word: 7,016 families; 59 relevant out of 79 labelled (0.75; 95% interval 0.64–0.83).

The remaining errors are other meanings of the word (the male bee, underwater drones, target drones) and inventions that mention a drone among several devices. Removing "drone" altogether would discard about 5,300 relevant families to avoid about 1,750 irrelevant ones.

In [13]:
import re
word = r"\bdrones?\b"

d = df_drone[df_drone["docdb_family_id"].isin(only("drone"))]
t = (d["appln_title"] + " " + d["appln_abstract"]).str.lower()
whole = set(d.loc[t.str.contains(word, regex=True), "docdb_family_id"])
print("Solo 'drone':", d["docdb_family_id"].nunique())
print("  con la parola intera:", len(whole))
print("  solo come pezzo di parola:", d["docdb_family_id"].nunique() - len(whole))

s = samples["Solo 'drone'"].copy()
s["parola_intera"] = (s["appln_title"] + " " + s["appln_abstract"]).str.lower().str.contains(word, regex=True)
print(s[s["llm_label"] != "incerto"].groupby("parola_intera")["llm_label"]
      .agg(n="size", pertinenti=lambda x: (x == "pertinente").sum()))


Solo 'drone': 8675
  con la parola intera: 7016
  solo come pezzo di parola: 1659
                n  pertinenti
parola_intera                
False          18           0
True           79          59


## 9. Terms tested for addition

**Question.** Does the initial list miss drones that other common terms would find?

Candidates not in the initial list: the abbreviation alone (" uav", "(uav"), the general form "unmanned aerial", "multirotor" / "multi-rotor" and "unmanned helicopter". First the count of families each would add (`non_ancora_nel_dataset`), then a sample of 50 for the groups that add enough families to matter.

**Rule (fixed before sampling).** A term is added only if at least 80% of the families it adds are relevant.

In [14]:
NEW_TERMS = (" uav", "(uav", "unmanned aerial", "multirotor", "multi-rotor", "unmanned helicopter")

in_dataset = set(df_drone["docdb_family_id"])
new_fams, rows = {}, []
for term in NEW_TERMS:
    q = (db.query(TLS201_APPLN.docdb_family_id)
         .select_from(TLS201_APPLN)
         .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
         .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
         .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
         .filter(keyword_condition((term,)))
         .distinct())
    f = set(patstat.df(q).iloc[:, 0])
    new_fams[term] = f
    rows.append({"termine": term, "famiglie": len(f), "non_ancora_nel_dataset": len(f - in_dataset)})

print("Famiglie nuove in totale:", len(set().union(*new_fams.values()) - in_dataset))
pd.DataFrame(rows)


Famiglie nuove in totale: 5873


,termine,famiglie,non_ancora_nel_dataset
0,uav,10039,2281
1,(uav,5350,89
2,unmanned aerial,149317,204
3,multirotor,282,124
4,multi-rotor,5619,1651
5,unmanned helicopter,2422,1644


In [15]:
def fetch_texts(fam_ids):
    q = (db.query(TLS201_APPLN.appln_id, TLS201_APPLN.docdb_family_id,
                  TLS202_APPLN_TITLE.appln_title, TLS203_APPLN_ABSTR.appln_abstract)
         .select_from(TLS201_APPLN)
         .outerjoin(TLS202_APPLN_TITLE, and_(
             TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
             func.lower(func.trim(TLS202_APPLN_TITLE.appln_title_lg)) == "en"))
         .join(TLS203_APPLN_ABSTR, and_(
             TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
             func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en"))
         .filter(TLS201_APPLN.docdb_family_id.in_([int(x) for x in fam_ids])))
    df = patstat.df(q)
    df.columns = ["appln_id", "docdb_family_id", "appln_title", "appln_abstract"]
    return df

groups_new = {
    "uav (sigla)": (new_fams[" uav"] | new_fams["(uav"]) - in_dataset,
    "multirotor / multi-rotor": (new_fams["multirotor"] | new_fams["multi-rotor"]) - in_dataset,
    "unmanned helicopter": new_fams["unmanned helicopter"] - in_dataset,
}

rng = np.random.default_rng(42)
rows_new, samples_new = [], {}
for name, fams in groups_new.items():
    drawn = rng.choice(sorted(fams), size=min(200, len(fams)), replace=False)
    texts = fetch_texts(drawn)
    sample = validate_sample(texts, n=50)
    samples_new[name] = sample
    lab = sample["llm_label"]
    k, n = int((lab == "pertinente").sum()), int(lab.isin(["pertinente", "non_pertinente"]).sum())
    low, high = wilson_interval(k, n)
    rows_new.append({"gruppo": name, "famiglie_nuove": len(fams),
                     "con_abstract_inglese_su_200": texts["docdb_family_id"].nunique(),
                     "etichettate": len(sample), "pertinenti": k,
                     "incerti": int((lab == "incerto").sum()),
                     "quota_pertinenti": round(k / n, 2) if n else None,
                     "IC95_min": round(low, 2), "IC95_max": round(high, 2)})
pd.DataFrame(rows_new)




Rows without a valid family or abstract: 0
Families labelled: 50 (new LLM calls: 0, already in cache: 50)
Rows without a valid family or abstract: 0
Families labelled: 50 (new LLM calls: 0, already in cache: 50)
Rows without a valid family or abstract: 0
Families labelled: 50 (new LLM calls: 0, already in cache: 50)


,gruppo,famiglie_nuove,con_abstract_inglese_su_200,etichettate,pertinenti,incerti,quota_pertinenti,IC95_min,IC95_max
0,uav (sigla),2320,194,50,25,1,0.51,0.37,0.64
1,multirotor / multi-rotor,1756,189,50,31,6,0.70,0.56,0.82
2,unmanned helicopter,1644,199,50,49,1,1.00,0.93,1.00


**Decision.**

| Term | New families | Relevant | Decision |
|---|---|---|---|
| unmanned helicopter | 1,644 | 1.00 (0.93–1.00) | **Added** |
| " uav" / "(uav" | 2,320 | 0.51 (0.37–0.64) | Rejected |
| multirotor / multi-rotor | 1,756 | 0.70 (0.56–0.82) | Rejected |
| unmanned aerial (general form) | 204 | not sampled | Not added: too few families |

All the terms tested together would add 5,873 families, about 3% of the dataset: the initial list was already close to saturation. The rejected terms do contain drones (roughly 1,200 families each); they are left out to avoid a similar number of irrelevant families.

## 10. Summary: from the initial to the final criteria

| | Initial | Final | Change |
|---|---|---|---|
| CPC | `B64U`, `B64C39/024`, `G08G5/57` | unchanged | none |
| IPC | `B64U` | `B64U`, `G08G5/57` | one group added (148 families) |
| Keywords | 22 terms, all as substrings | 14 terms as substrings + "drone" as a whole word | 1 added, 8 removed, 1 corrected |

Final lists, to be used in `1-DataPreparation`:

```python
DRONE_CPC_CODES = ("B64U", "B64C39/024", "G08G5/57")
DRONE_IPC_CODES = ("B64U", "G08G5/57")

DRONE_KEYWORDS = (
    # Names of the vehicle
    "unmanned aerial vehicle", "unmanned aerial system", "unmanned aircraft",
    "unmanned air vehicle", "unmanned helicopter", "uncrewed aircraft",
    "remotely piloted aircraft",
    # Configurations
    "quadcopter", "quadrotor", "quadrocopter", "multicopter",
    "hexacopter", "octocopter", "micro air vehicle",
)
DRONE_KEYWORD_PATTERNS = (r"\bdrones?\b",)   # whole word, not substring
```

Keyword changes: "unmanned helicopter" added; "unmanned aircraft system" removed (redundant); the seven "uav + technology/configuration" terms removed (66 exclusive families in total, and the abbreviation alone is 51% relevant); "drone" searched as a whole word.

**What these checks do not cover.**

- About 5,800 families retrieved only by the smaller keywords were not sampled.
- Undecided cases still have to be labelled by hand.
- The whole-word test for "drone" was run on English texts.
- Samples of 50–100 families give wide intervals; the labels are those of the LLM.
- Recall is observed only within what the query retrieves: drones that no criterion finds are not seen.